# S09 - Lab exercice 02
## Yago Ramos Sánchez


The attached files are a collection of tweets labelled with sentiment in 3 categories:

sentiments = { "LABEL_0": "Bearish", "LABEL_1": "Bullish", "LABEL_2": "Neutral" }
Train a LSTM network with the training file. Validate the trained model with the valid file.

In [22]:
# Libraries importation

import torch 
import torch.nn as nn
import pandas as pd
import numpy as np
import re
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

In [26]:
# Text preprocessing

def clean_text(text):
    """Removing URLs, mentions, hashtags, punctuation and converting to lowercase."""
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'@\w+', '', text)  # Remove mentions
    text = re.sub(r'#\w+', '', text)  # Remove hashtags
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = text.lower()  # Convert to lowercase
    return text

# Apply cleaning to the text column
train_df[text_column] = train_df[text_column].apply(clean_text)
valid_df[text_column] = valid_df[text_column].apply(clean_text)

# Hyperparameters for text processing
MAX_WORDS = 10000  # Maximum number of words in the vocabulary
MAX_LEN = 100  # Maximum length of input sequences

# Building the vocabulary
all_words = ' '.join(train_df[text_column]).split()
word_counts = Counter(all_words)

# Keep only the most common words
most_common_words = word_counts.most_common(MAX_WORDS - 2)  # Reserve 2 for PAD and UNK
vocab = {word: idx + 2 for idx, (word, _) in enumerate  (most_common_words)}  # Start indexing from 2

def text_to_sequence(text):
    tokens = text.split()
    # Map words to ints, using 1 for unknown words
    sequence = [vocab.get(token, 1) for token in tokens]
    # Truncate if the worg is too long
    if len(sequence) > MAX_LEN:
        sequence = sequence[:MAX_LEN] 
    # Pad with zeros if the sequence is too short
    else:
        sequence += [0] * (MAX_LEN - len(sequence))
    return sequence

# Custom PyTorch Dataset
class TweetDataset(Dataset):
    """Custom Dataset to handle our text sequences and labels."""
    def __init__(self, df, vocab, max_len):
        self.sequences = [text_to_sequence(t, vocab, max_len) for t in df['clean_text']]
        self.labels = df[label_col].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Convert lists to PyTorch tensors
        seq = torch.tensor(self.sequences[idx], dtype=torch.long)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return seq, label

# Create DataLoaders to feed data in batches
BATCH_SIZE = 32
# Ensure the column expected by TweetDataset exists
if "clean_text" not in train_df.columns:
    train_df["clean_text"] = train_df[text_column].apply(clean_text)
if "clean_text" not in valid_df.columns:
    valid_df["clean_text"] = valid_df[text_column].apply(clean_text)

# Align variable/function names used inside TweetDataset
label_col = label_column

def text_to_sequence(text, vocab=vocab, max_len=MAX_LEN):
    tokens = text.split()
    sequence = [vocab.get(token, 1) for token in tokens][:max_len]
    sequence += [0] * (max_len - len(sequence))
    return sequence

train_dataset = TweetDataset(train_df, vocab, MAX_LEN)
valid_dataset = TweetDataset(valid_df, vocab, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# Define the LSTM-based model for sentiment analysis

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers=1, dropout=0.5):
        super(SentimentLSTM, self).__init__()
        
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(input_size=embed_dim, hidden_size=hidden_dim, 
                            num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):

        # x shape: (batch_size, seq_length)
        embedded = self.embedding(x) 
        
        # Pass through LSTM
        lstm_out, (hidden, cell) = self.lstm(embedded)
        
        # We only need the output from the final time step for classification
        final_state = lstm_out[:, -1, :] 
        
        out = self.dropout(final_state)
        logits = self.fc(out)
        return logits

# Initialize Model, Loss function, and Optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

EMBED_DIM = 128
HIDDEN_DIM = 64
NUM_CLASSES = 3

model = SentimentLSTM(vocab_size=MAX_WORDS, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM, num_classes=NUM_CLASSES)
model = model.to(device)

# CrossEntropyLoss automatically applies Softmax
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Using device: cpu
